In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Add project root (parent of this scripts/ folder) to sys.path once
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.models_and_vars import load_env_vars, ensure_nltk_path, init_spacy_and_emo
from src.data_processing import subset_by_group_labels
from src.fmtools import (
    monitor_memory,
    draw_fmnts,
    make_emo_flowers,
    all_fmns_features_df,
    verify_and_report_nodes
)
from src.tfmtools import extract_tfm_structural_features, build_tfmns_by_label
from src.io import export_df, export_whole_fmn_multiple_custom

pd.set_option('display.max_rows', 100)

In [ ]:
# you do not need to run these download commands every time. nltk.download()
# saves these data packages to your local file system in the nltk_data directory.
# Once they are downloaded, they are available for NLTK to use in any session.

'''nltk.download('wordnet')
nltk.download('omw-1.4') # for italian and multilingual support, if needed
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng') # for word tokenization
nltk.download('averaged_perceptron_tagger') # for POS tagging
'''


# Environment and models settings

In [ ]:
# Set language once; everything else adapts
language = "italian"  # or "english"
env_file_path = "../env"

# Execute setup
load_env_vars(env_file_path, language=language)
ensure_nltk_path()

nlp, emo = init_spacy_and_emo(language)

# Also preload both models for explicit use
nlp_en, _ = init_spacy_and_emo("english")
nlp_it, _ = init_spacy_and_emo("italian")

# fmn settings

In [ ]:
# Dynamic output paths system for pipelines
base_output_path = '../out'
project_name = 'merged'
formatted_data_dir = 'formatting'
use_prelemmatized_brysbaert = True # A lot faster (highly suggested, but needs pre-lemmatized data)
# Set the type of fmn to use: 'bfmns' or 'tfmns'
# (this is used for the folder name, not for the file names)
fmn_type = 'tfmns'
# Keywords that we want to check for in the textual data. If not empty,
# semantic frame plots will be created for these keywords. Otherwise,
# the full FMNs will be visualized.
keywords = []
keywords = ['matematica', 'ansia', 'stress', 'depressione']

# Types of output: aggregated and individual (entire dataset vs per participant FMNs)
# Choose any subset: {'aggregated'}, {'individual'}, or both
# How to use:
# Only aggregated: methods = {'aggregated'}
# Only individual: methods = {'individual'}
# Both: methods = {'aggregated', 'individual'}
ALLOWED_METHODS = {'aggregated', 'individual'}
methods = {'aggregated'}
methods = {m.strip().lower() for m in (methods if isinstance(methods, (list, set, tuple)) else [methods])}
methods = methods & ALLOWED_METHODS
if not methods:
    raise ValueError("No valid methods selected. Use 'aggregated' and/or 'individual'.")

# Groups to subset the data (if needed). Use labels and .startswith() condition
# from the 'participant_id' column to create groups.
group_name = 'physics'  # fmn name when using groups
if group_name:
    fmn_type = f'{group_name}_{fmn_type}'

participant_prefixes = ['fisica']

# If dfs_and_group_labels is longer than 1, merge the data from the different paths.
dfs_and_group_labels = []
df1 = pd.read_csv('../data/extracted_data_2025/free_text.csv')
dfs_and_group_labels = [
    (df1, participant_prefixes),
]

# Replace cells containing only a period with NaN
for df, _ in dfs_and_group_labels:
    df['text'] = df['text'].replace(r'^\s*\.\s*$', np.nan, regex=True)

# print all unique participants
all_participants = set()
for df, group_labels in dfs_and_group_labels:
    all_participants.update(df['participant_id'].unique())
print(f"All unique participants across datasets: {len(all_participants)}")
print(f"Unique participants labels: {all_participants}\n")

# Resolve output directories
if methods == {'aggregated'}:
    files_output_dir = f'{base_output_path}/{project_name}/aggregated/{fmn_type}'
    data_output_dir = f'{base_output_path}/{project_name}/aggregated/{formatted_data_dir}/{fmn_type}'
if methods == {'individual'}:
    files_output_dir = f'{base_output_path}/{project_name}/individual/{fmn_type}'
    data_output_dir = f'{base_output_path}/{project_name}/individual/{formatted_data_dir}/{fmn_type}'

### Memory Optimization Tips

The following optimizations have been implemented to prevent RAM issues:

1. **Batch Processing**: FMNs are built in small batches (5-10 at a time) with garbage collection between batches
2. **Memory Monitoring**: `monitor_memory()` function tracks RAM usage throughout the pipeline
3. **Optional Visualization**: Drawing and emotional flowers are commented out by default - uncomment only if needed
4. **Cleanup**: Matplotlib figures are properly closed after each use
5. **Reduced Batch Sizes**: Smaller batch sizes for memory-intensive operations

To enable visualizations, uncomment the relevant sections but be aware of memory usage.

### Data loading and pre-processing

In [ ]:
# Subset by group labels if multiple dataframes and labels provided, otherwise just merge):
final_df = subset_by_group_labels(dfs_and_group_labels, col='participant_id')
# Replace cells containing only a period with NaN
final_df['text'] = final_df['text'].replace(r'^\s*\.\s*$', np.nan, regex=True)

# Group all texts from the final dataframe into a unique string
all_texts = ' '.join(final_df['text'].dropna().tolist())
# Print to check  correct subsetting and merging
print(f"\n{all_texts}")

In [ ]:
# Export long and categorized dataframes (edit path as needed)
# Also dynamically name files based if method is aggregated or individual
if len(dfs_and_group_labels) == 1:
    export_df(final_df, data_output_dir, filename=f'{group_name}_texts')

elif len(dfs_and_group_labels) > 1:
    export_df(final_df, data_output_dir, filename=f'{group_name}_texts')

## Run the pipeline on preprocessed data

### Aggregated pipeline

In [ ]:
# Method to execute: a single aggregated FMN from all data
if 'aggregated' in methods:
    # Monitor initial memory usage
    print("Starting aggregated pipeline...\n")

    # build the aggregated FMN and print it
    all_texts_fmn = emo.formamentis_network(text=all_texts, target_word=None, semantic_enrichment=['synonyms'], multiplex=False)
        
    monitor_memory()

    export_whole_fmn_multiple_custom(
    fmnts=[all_texts_fmn],
    save_path=files_output_dir,
    filename=f'{group_name}',
    custom_valences=None,
    language=emo.language
    )

    monitor_memory()

    # Load Brysbaert concreteness dataset
    df_brysbaert = pd.read_csv('../data/brysbaert_concreteness_dataset_eng_ita/brysbaert_concreteness_dataset_eng_ita.csv')

    # Draw and save semantic frame plots and emo flowers for each keyword
    # for each keyword in keywords extract it, then draw the semantic
    # frame plot, save it and show it in the notebook
    layout = "edge_bundling"
    if len(keywords) > 0:
        for keyword in keywords:
            semantic_frame = emo.extract_word_from_formamentis(all_texts_fmn, keyword)
            if not hasattr(semantic_frame, "vertices") or len(semantic_frame.vertices) == 0:
                print(f"Skipping '{keyword}': not in network or empty semantic frame.")
                continue
            out_dir = f'{base_output_path}/{files_output_dir}/visualizations'
            filename = f'tfmn_{group_name}_{keyword}_{layout}_semantic_frame'
            draw_fmnts(
                fmnts={group_name: semantic_frame},
                out_dir=out_dir,
                layout=layout,
                highlight=[],
                thickness=1,
                ax=None,
                emo=emo,
                custom_valences=None,
                filename_base=filename,
            )
            make_emo_flowers(
                fmnts={f'{group_name}_{keyword}': semantic_frame},
                save_path=out_dir,
                filename=f'{group_name}_{keyword}_emo_flower',
                emo=emo,
                batch_size=1
            )

            if use_prelemmatized_brysbaert:
                # Analyze concreteness of the semantic frame (fast, pre-lemmatized datasets)
                try:
                    result = emo.analyze_semantic_frame_concreteness_fast(
                        semantic_frame,
                        seed=None,
                        save_json_path=f'{base_output_path}/{files_output_dir}/analysis/{group_name}_{keyword}_concreteness_analysis.json',
                        save_figure_path=f'{base_output_path}/{files_output_dir}/visualizations/{group_name}_{keyword}_concreteness_analysis.png',
                        plot=True
                    )
                    print(result['p_value'])
                    print(f"Concreteness analysis for '{keyword}' saved successfully.")
                except Exception as e:
                    print(f"Error analyzing concreteness for '{keyword}': {e}")
            
            else:
                # Analyze concreteness of the semantic frame
                try:
                    result = emo.analyze_semantic_frame_concreteness(
                        semantic_frame,
                        df_brysbaert,
                        seed=None,
                        save_json_path=f'{base_output_path}/{files_output_dir}/analysis/{group_name}_{keyword}_concreteness_analysis.json',
                        save_figure_path=f'{base_output_path}/{files_output_dir}/visualizations/{group_name}_{keyword}_concreteness_analysis.png',
                        plot=True
                    )
                    print(result['p_value'])
                    print(f"Concreteness analysis for '{keyword}' saved successfully.")
                except Exception as e:
                    print(f"Error analyzing concreteness for '{keyword}': {e}")

        monitor_memory()

    '''valence_stats = fmn_valence.fmn_valence_stats(merged_fmn, custom_valences)
    print("Valence stats:", valence_stats)

    # Inspect the merged FMN
    inspect_fmn(
        fmn=all_texts_fmn,
        participant=group_name,
        all_features={group_name: valence_stats},
        emo=emo
    )'''

    # Check for presence of each keyword in the fmn
    verify_and_report_nodes(
        fmns={group_name: all_texts_fmn},
        required=keywords
    )
        
    # Compute and export structural features for the aggregated FMN
    aggregated_features = extract_tfm_structural_features(
        fmnts={group_name: all_texts_fmn},
        save_path=f'{base_output_path}/{files_output_dir}/structural_features',
        emo=emo,
        batch_size=1,
        persist=True
    )

    # Convert features to DataFrame and export
    aggregated_features_df = all_fmns_features_df(aggregated_features, orient="participants")
    export_df(aggregated_features_df, f'{base_output_path}/{files_output_dir}/structural_features', filename=f'{group_name}_structural_features')


### Individual pipeline

In [ ]:
# Method to execute: a single aggregated FMN from all data
if 'individual' in methods:
    print("Starting individual pipeline...\n")

    merged_fmn = emo.formamentis_network(text=all_texts, target_word=None, semantic_enrichment=['synonyms'], multiplex=False)

    tfmns_by_label = build_tfmns_by_label(
        texts_by_label=texts_by_participant,
        emo=emo,
        keywords=keywords,
        labels=list(participant_ids),
        batch_size=10
    )

    # Single flat output folder for all images
    vis_out_dir = f'{base_output_path}/{files_output_dir}/visualizations'
    os.makedirs(vis_out_dir, exist_ok=True)

    monitor_memory()

    for tfmn_title, tfmn in tfmns_by_label.items():
        # Export networks (also switch to the "individual" output root)
        export_whole_fmn_multiple_custom(
            fmnts=[tfmn],
            save_path=f'{base_output_path}/{files_output_dir}',
            filename=f'{group_name}_{tfmn_title}',
            custom_valences=None,
            language=emo.language
        )

        monitor_memory()

        layout = "edge_bundling"
        if len(keywords) > 0:
            for keyword in keywords:
                semantic_frame = emo.extract_word_from_formamentis(tfmn, keyword)
                # Save directly in the shared visualizations folder (no subfolders)
                draw_fmnts(
                    fmnts={tfmn_title: semantic_frame},
                    out_dir=vis_out_dir,
                    layout=layout,
                    highlight=[],
                    thickness=1,
                    ax=None,
                    custom_valences=None,
                    filename_base=f'{tfmn_title}_{keyword}_{layout}_semantic_frame',
                )
        else:
            # Full TFMN image, flat in the same folder
            draw_fmnts(
                fmnts={tfmn_title: tfmn},
                out_dir=vis_out_dir,
                layout=layout,
                highlight=[],
                thickness=1,
                ax=None,
                emo=emo,
                custom_valences=None,
                filename_base=f'{tfmn_title}_{layout}',
            )

        monitor_memory()


        '''valence_stats = fmn_valence.fmn_valence_stats(merged_fmn, custom_valences)
        print("Valence stats:", valence_stats)

        # Inspect the merged FMN
        inspect_fmn(
            fmn=tfmn,
            participant=group_name,
            all_features={group_name: valence_stats},
            emo=emo
        )'''

        # Check for presence of each keyword in the fmn
        verify_and_report_nodes(
            fmns={group_name: tfmn},
            required=keywords
        )

    # Features -> also use the "individual" data folder (no per-title dirs)
    all_features = extract_tfm_structural_features(
        fmnts={tfmn_title: tfmn},
        save_path=f'{base_output_path}/{files_output_dir}/{tfmn_title}_structural_features',
        emo=emo,
    )